# Multi-Object Tracking Under Occlusion (OcclusionDeepSORT)
### End-to-End Benchmark Evaluation, Architecture & Failure Analysis

This notebook demonstrates the complete production pipeline for tracking individuals across severe visual occlusions, crowded crossing paths, and temporary disappearances.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# Ensure project root is in sys.path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.detectors.yolo_detector import YOLOPersonDetector
from src.reid.feature_extractor import ReIDFeatureExtractor
from src.tracker.occlusion_tracker import OcclusionTracker
from src.evaluation.mot_evaluator import MOTEvaluator
from src.evaluation.failure_analyzer import FailureAnalyzer

print("All modules imported successfully!")

## 1. Inspect Tracker Configurations (Baseline vs Tuned)
We compare **Baseline SORT** (minimal buffering, pure spatial matching, immediate new ID generation) against **Tuned OcclusionDeepSORT** (lost track buffer, appearance memory gallery, occlusion freeze, and Re-ID recovery).

In [ ]:
with open(project_root / "configs" / "baseline_config.yaml", "r") as f:
    baseline_cfg = yaml.safe_load(f)
with open(project_root / "configs" / "tuned_config.yaml", "r") as f:
    tuned_cfg = yaml.safe_load(f)

comparison_params = pd.DataFrame([
    {"Parameter": "max_lost_age (Lost Buffer Frames)", "Baseline": baseline_cfg['tracker']['max_lost_age'], "Tuned": tuned_cfg['tracker']['max_lost_age']},
    {"Parameter": "appearance_weight (Re-ID α)", "Baseline": baseline_cfg['tracker']['appearance_weight'], "Tuned": tuned_cfg['tracker']['appearance_weight']},
    {"Parameter": "gallery_size (Appearance Memory)", "Baseline": baseline_cfg['tracker']['gallery_size'], "Tuned": tuned_cfg['tracker']['gallery_size']},
    {"Parameter": "enable_lost_reid (Re-ID on Reappearance)", "Baseline": baseline_cfg['tracker']['enable_lost_reid'], "Tuned": tuned_cfg['tracker']['enable_lost_reid']},
    {"Parameter": "enable_occlusion_freeze (Protect Gallery)", "Baseline": baseline_cfg['tracker']['enable_occlusion_freeze'], "Tuned": tuned_cfg['tracker']['enable_occlusion_freeze']}
])
comparison_params

## 2. Experimental Benchmark Results
Let us examine the MOT challenge evaluation results on the standardized occlusion benchmark sequence (`MOT17-SYNTH-OCCLUSION`).

In [ ]:
evaluator = MOTEvaluator(iou_threshold=0.5)
gt_path = project_root / "data" / "MOT17-SYNTH-OCCLUSION" / "gt" / "gt.txt"
base_res = project_root / "results" / "baseline_results.txt"
tuned_res = project_root / "results" / "tuned_results.txt"

base_eval = evaluator.evaluate_sequence(gt_path, base_res, seq_name="Baseline")
tuned_eval = evaluator.evaluate_sequence(gt_path, tuned_res, seq_name="Tuned")

print("=== BASELINE RESULTS ===")
print(base_eval["rendered_table"])
print("\n=== TUNED RESULTS ===")
print(tuned_eval["rendered_table"])

## 3. Comparative Metric Visualizations
We visualize the reduction in ID switches and gain in IDF1.

In [ ]:
summary_img = project_root / "results" / "comparison_metrics.png"
if summary_img.exists():
    img = plt.imread(str(summary_img))
    plt.figure(figsize=(16, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Tracker Benchmark Metrics: Baseline vs Tuned OcclusionSORT", fontsize=14)
    plt.show()

## 4. Automated Failure Case & Root Cause Analysis
Using the `FailureAnalyzer`, we inspect the exact frame and cause that led to identity switches in the baseline tracker.

In [ ]:
analyzer = FailureAnalyzer()
fail_analysis = analyzer.analyze(gt_path, base_res)
print(f"Total Identified ID Switches: {fail_analysis['total_switches']}")
print("\nRoot Cause Breakdown:")
print(fail_analysis["cause_breakdown"])
print("\nDetailed Incident Log:")
for inc in fail_analysis["incidents"]:
    print(f"Frame {inc['frame']}: GT ID {inc['gt_id']} switched from {inc['old_pred_id']} -> {inc['new_pred_id']} | Cause: {inc['primary_cause']} ({inc['details']})")

## 5. Generated Video Artifacts
The pipeline produces complete MP4 tracking videos with bounding boxes, persistent IDs, trajectory tails, and HUD dashboards:
- `visualizations/tuned_tracking_demo.mp4` (Tuned Occlusion Tracker demo)
- `visualizations/baseline_tracking_demo.mp4` (Baseline tracker demo)
- `visualizations/vtest_real_tracked.mp4` (Real-world surveillance pedestrian tracking using pretrained YOLOv8 + Re-ID)